# DDPM Study Notes — Likelihood through Equation (7)

*Transcribed from handwritten notebook pages 1–36. Covers Ho, Jain & Abbeel (2020), "Denoising Diffusion Probabilistic Models", through Eq. (7) on page 3 — the end of Section 2, just before Section 3.*

---

## 1. Likelihood

Flip a coin 10 times and observe 7 heads, 3 tails. Call $p$ the probability of heads. We do not know $p$. The question is which value of $p$ best explains what we saw.

For any candidate $p$, we can compute how probable our observation would be:

$$P(\text{7 heads in 10 flips} \mid p) = \binom{10}{7} p^{7} (1-p)^{3}$$

Now, try several different values of $p$ and find the value for which it is the maximum; this is the **maximum likelihood estimate**.

| $p$ | $P$ |
|:---|:---|
| 0.1 | 0.0000087 |
| 0.3 | 0.0090 |
| 0.5 | 0.117 |
| 0.7 | 0.267 |
| 0.9 | 0.0574 |

The value $p = 0.7$ makes the observed data most probable. That is the maximum likelihood estimate.

---

## 2. Many data points at once

With $n$ independent observations $x_1, x_2, \ldots, x_n$, the probability of seeing all of them is the product of the individual probabilities:

$$L(\theta) = P(x_1 \mid \theta) \times P(x_2 \mid \theta) \times \cdots \times P(x_n \mid \theta)$$

$$L(\theta) = \prod_{i=1}^{n} P(x_i \mid \theta)$$

Here $\theta$ stands for whatever parameters the model has; for a normal distribution, $\theta$ means the pair $(\mu, \sigma)$.

### Why do we take the logarithm of the likelihood?

For hundreds of observations, we are multiplying hundreds of probabilities, producing a very small number that the computer cannot represent.

We fix this by turning products into sums:

$$\log L(\theta) = \sum_{i=1}^{n} \log P(x_i \mid \theta)$$

As $\log$ is monotonically increasing, whichever $\theta$ maximizes $L$ also maximizes $\log L$.

Optimizers minimize rather than maximize, so we flip the sign and minimize the **negative log-likelihood**:

$$\mathrm{NLL}(\theta) = -\sum_{i=1}^{n} \log P(x_i \mid \theta)$$

Maximizing likelihood and minimizing NLL are the same:

$$\hat{\theta} = \arg\max_{\theta} L(\theta) = \arg\min_{\theta} \mathrm{NLL}(\theta)$$

---

## 3. Surprisal and Shannon information content

The quantity $-\log P(x)$ has a name in information theory: **surprisal**, or Shannon information content.

Suppose we want some function $S(p)$ measuring how surprising an event of probability $p$ is. We can see that $-\log p$ is the most reasonable one for representing surprisal.

**If an event is certain, we expect zero surprise.**

$$S(1) = -\log 1 = 0$$

If an event is rare, we expect the surprise to increase; the more common it is, the less surprising it will be and so $S$ will decrease.

**If two uncorrelated things both happen, total surprise is the sum.** Their joint probability is $p \cdot q$, so

$$S(p \cdot q) = S(p) + S(q)$$

If it is very rare, $-\log 0 \to \infty$, so we have very high surprise.

---

## 4. The forward process in diffusion modeling

In diffusion modeling, we start with a clean image $x_0$. Then we add random noise to the image cumulatively. This is called the **forward process** $q$.

$$q(x_1 \mid x_0) \rightarrow \text{the conditional density of } x_1 \text{ given } x_0$$

All the noisy images from $x_1$ to $x_T$ are called **latents**.

All CIFAR-10 images have 32 pixels for width, 32 pixels for height and 3 colour channels. Thus, each image has

$$32 \times 32 \times 3 = 3072 \text{ numbers per image}$$

Also, as we add noise over 1000 steps, there are an additional $3072 \times 1000 = 3{,}072{,}000$ numbers for all the 1000 latents.

### The conditional density

$q(x_1 \mid x_0)$ is the conditional density function of the random variable $x_1$, evaluated at the point $x_1$, given that the conditioning variables take the value $x_0$:

$$q(x_1 \mid x_0) = \mathcal{N}\!\left(x_1;\ \sqrt{1-\beta_1}\, x_0,\ \beta_1 I\right)$$

Here, $\beta_t$ is the variance of the noise added at step $t$.

Ho et al. used a **linear schedule** from $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$ over $T = 1000$ steps.

$$q(x_t \mid x_{t-1}) := \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right) \qquad (2)$$

Here, "$:=$" means "defined as". Written out explicitly (per coordinate):

$$q(x_t \mid x_{t-1}) = \frac{1}{\sqrt{2\pi \beta_t}} \exp\!\left(-\frac{\left(x_t - \sqrt{1-\beta_t}\, x_{t-1}\right)^2}{2\beta_t}\right)$$

And for the whole chain:

$$q(x_{1:T} \mid x_0) := \prod_{t=1}^{T} q(x_t \mid x_{t-1}) \qquad (2)$$

We use the **Markov assumption** here: each step depends on the immediately previous step and not on any other steps preceding this previous step.

---

## 5. Reading the Gaussian notation

$\mathcal{N}(x; \mu, \sigma^2)$ is the probability of observing $x$ under a Gaussian distribution with mean $\mu$ and variance $\sigma^2$. So

$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right)$$

means: given $x_{t-1}$, the distribution of $x_t$ is Gaussian with mean $\sqrt{1-\beta_t}\, x_{t-1}$ and variance $\beta_t$.

Here $\beta_t \in (0,1)$ — read "$\beta_t$ belongs to the set $(0,1)$". $\beta_t$ represents how much noise is added at step $t$, and is called the **noise schedule**.

### Why did the authors choose the coefficients $\sqrt{1-\beta_t}$ and $\sqrt{\beta_t}$?

$$x_t \sim \mathcal{N}\!\left(\underbrace{\sqrt{1-\beta_t}\, x_{t-1}}_{\text{mean } \mu},\ \underbrace{\beta_t I}_{\sigma^2}\right)$$

In reparameterized form:

$$x_t = \sqrt{1-\beta_t}\, x_{t-1} + \sqrt{\beta_t}\, \epsilon$$

Taking variances, with $\epsilon \sim \mathcal{N}(0, I)$:

$$\mathrm{Var}(x_t) = (1-\beta_t)\,\mathrm{Var}(x_{t-1}) + \beta_t \cdot 1$$

The data is typically normalized to have unit variance before training, as pixel values are scaled to $[-1, 1]$ and standardized. So $\mathrm{Var}(x_0) \approx 1$ is a reasonable starting point.

Given that, if $\mathrm{Var}(x_{t-1}) = 1$:

$$\mathrm{Var}(x_t) = (1-\beta_t) + \beta_t = 1$$

By induction, every step preserves unit variance. Without this scaling, the variance could grow at every step.

---

## 6. $p_\theta(\cdot)$ — the reverse process

The **reverse process** tries to remove noise. Parameters $\theta$ are learned; $\theta$ are learnable weights of our neural network.

In the reverse process, we go from the noisy image or latent at step 1000 to the step-1 image, and then to the initial state or image that we started from.

### $p_\theta(x_{0:T})$ or $p_\theta(x_0, x_1, x_2, \ldots, x_T)$

What is the probability the reverse process produces this **exact sequence** $x_T, x_{T-1}, \ldots, x_2, x_1, x_0$? The probability that we start with the noisy image $x_T$, then the partly denoised image $x_{T-1}$, ..., then the denoised image $x_1$, and then the original image $x_0$.

### $p_\theta(x_0)$

What is the probability the process ends up at this photo, **by any route whatever**? It does not care what happened in the middle.

$$p_\theta(x_0) = \int p_\theta(x_0, x_1, \ldots, x_T)\, dx_1\, dx_2 \cdots dx_T$$

$$p_\theta(x_0) = \int\!\!\int \cdots \int p_\theta(x_{0:T})\, dx_1\, dx_2 \cdots dx_T$$

Here, we have 1000 nested integral signs and 1000 differentials $dx_1$ through $dx_T$.

For each image, the number of pixels is $32 \times 32 \times 3 = 3072$. Hence, one integral sign is just shorthand for 3072 nested ones. Writing $dx_1$ for one latent (noisy) image means

$$dx_1^{(1)},\ dx_1^{(2)},\ \ldots,\ dx_1^{(3072)}$$

> **Note:** $\displaystyle \int p_\theta(x_0, x_1, \ldots, x_T)\, dx_0\, dx_1 \cdots dx_T = 1$

### $x_0 \sim q(x_0)$

The real image $x_0$ is a sample from the **data distribution** $q(x_0)$. This $q$ is **not** the same as $q(x_t \mid x_{t-1})$. Here $q(x_0)$ is the data distribution.

---

## 7. Factorizing $p_\theta(x_{0:T})$ — Equation (1)

By the chain rule of probability:

$$p_\theta(x_{0:T}) = P(x_T)\, P(x_{T-1} \mid x_T)\, P(x_{T-2} \mid x_{T-1}, x_T) \cdots P(x_0 \mid x_1, x_2, \ldots, x_T)$$

Now we will use the **Markov assumption**: each step looks only at the one immediately preceding it.

$$P(x_{t-1} \mid x_t, x_{t+1}, \ldots, x_T) = P(x_{t-1} \mid x_t)$$

So,

$$p_\theta(x_{0:T}) = P(x_T)\, p_\theta(x_{T-1} \mid x_T)\, p_\theta(x_{T-2} \mid x_{T-1}) \cdots p_\theta(x_0 \mid x_1)$$

Thus, we can write

$$p_\theta(x_{0:T}) = p(x_T) \prod_{t=1}^{T} p_\theta(x_{t-1} \mid x_t) \qquad (1)$$

### Is $p_\theta(x_0)$ the same as $P(x_0 \mid \theta)$?

Yes, but —

- $p_\theta(x_0)$ → the distribution **indexed by** $\theta$, evaluated at $x_0$. Assumes $\theta$ is not random.
- $P(x_0 \mid \theta)$ → the probability of $x_0$ **given** $\theta$. Presumes $\theta$ has its own distribution.

In this paper, the authors used $p_\theta(x_0)$. $x_0$ is a fixed photo from the dataset, and we are adjusting $\theta$ to push the likelihood up.

---

## 8. The reverse transition $p_\theta(x_{t-1} \mid x_t)$

$$p_\theta(x_{t-1} \mid x_t) := \mathcal{N}\!\left(x_{t-1};\ \mu_\theta(x_t, t),\ \Sigma_\theta(x_t, t)\right)$$

One denoising step is a Gaussian, centered wherever the neural network says, with a spread the network also specifies.

Feed in the noisy image $x_t$ and the number $t$, get back a mean image. The $\theta$ subscript marks them as the learned parts.

But in this paper, later, $\Sigma_\theta(x_t, t) = \sigma_t^2 I$ with $\sigma_t^2$ set to $\beta_t$. So only $\mu_\theta$ carries $\theta$ in practice. So, we only need to learn $\mu_\theta(x_t, t)$ as found from the neural network used to model the denoising process.

$$p_\theta(x_{t-1} \mid x_t) := \mathcal{N}\!\left(x_{t-1};\ \mu_\theta(x_t, t),\ \Sigma_\theta(x_t, t)\right) \qquad (1)$$

### The matching forward factorization

$$q(x_{1:T} \mid x_0) = \prod_{t=1}^{T} q(x_t \mid x_{t-1})$$

Expanding by the chain rule:

$$q(x_{1:T} \mid x_0) = q(x_1 \mid x_0)\, q(x_2 \mid x_1, x_0)\, q(x_3 \mid x_2, x_1, x_0) \cdots q(x_T \mid x_{T-1}, \ldots, x_1, x_0)$$

Now, use the Markov assumption:

$$q(x_t \mid x_{t-1}, x_{t-2}, \ldots, x_0) = q(x_t \mid x_{t-1})$$

So,

$$q(x_{1:T} \mid x_0) = q(x_1 \mid x_0)\, q(x_2 \mid x_1)\, q(x_3 \mid x_2) \cdots q(x_T \mid x_{T-1})$$

$$\Rightarrow \quad q(x_{1:T} \mid x_0) = \prod_{t=1}^{T} q(x_t \mid x_{t-1}) \qquad (2)$$

---

## 9. Expectation notation and Jensen's inequality

### $\mathbb{E}_q[\cdot]$

The notation $\mathbb{E}_q[f(x)]$ means the expected value of $f(x)$ when $x$ is drawn from distribution $q$. In practice, this is average $f(x)$ over many draws from $q$:

$$\mathbb{E}_q[f(x)] = \int q(x) \cdot f(x)\, dx$$

### Jensen's inequality

$$\log \mathbb{E}[Z] \geq \mathbb{E}[\log Z]$$

---

## 10. Equation (3) — the variational bound

$$\mathbb{E}\!\left[-\log p_\theta(x_0)\right] \leq \mathbb{E}_q\!\left[-\log \frac{p_\theta(x_{0:T})}{q(x_{1:T} \mid x_0)}\right] = \mathbb{E}_q\!\left[-\log p(x_T) - \sum_{t \geq 1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_t \mid x_{t-1})}\right] =: L \qquad (3)$$

### Derivation

$$\log p_\theta(x_0) = \log \int p_\theta(x_{0:T})\, dx_{1:T}$$

Multiply and divide by $q(x_{1:T} \mid x_0)$:

$$\log p_\theta(x_0) = \log \int q(x_{1:T} \mid x_0) \cdot \frac{p_\theta(x_{0:T})}{q(x_{1:T} \mid x_0)}\, dx_{1:T}$$

That integral is exactly an expectation, since

$$\mathbb{E}_{q(x_{1:T} \mid x_0)}\!\left[\frac{p_\theta(x_{0:T})}{q(x_{1:T} \mid x_0)}\right] = \int q(x_{1:T} \mid x_0)\, \frac{p_\theta(x_{0:T})}{q(x_{1:T} \mid x_0)}\, dx_{1:T}$$

So,

$$\log p_\theta(x_0) = \log \mathbb{E}_{q(x_{1:T} \mid x_0)}\!\left[\frac{p_\theta(x_{0:T})}{q(x_{1:T} \mid x_0)}\right] \geq \mathbb{E}_{q(x_{1:T} \mid x_0)}\!\left[\log \frac{p_\theta(x_{0:T})}{q(x_{1:T} \mid x_0)}\right]$$

### Expanding the log-ratio

From (1), $p_\theta(x_{0:T}) = p(x_T) \prod_{t=1}^{T} p_\theta(x_{t-1} \mid x_t)$.

From (2), $q(x_{1:T} \mid x_0) := \prod_{t=1}^{T} q(x_t \mid x_{t-1})$.

So,

$$\log \frac{p_\theta(x_{0:T})}{q(x_{1:T} \mid x_0)} = \log \frac{p(x_T) \prod_{t=1}^{T} p_\theta(x_{t-1} \mid x_t)}{\prod_{t=1}^{T} q(x_t \mid x_{t-1})}$$

$$= \log\!\left(p(x_T) \prod_{t=1}^{T} p_\theta(x_{t-1} \mid x_t)\right) - \log\!\left(\prod_{t=1}^{T} q(x_t \mid x_{t-1})\right)$$

$$= \log p(x_T) + \sum_t \log p_\theta(x_{t-1} \mid x_t) - \sum_t \log q(x_t \mid x_{t-1})$$

$$= \log p(x_T) + \sum_t \left(\log p_\theta(x_{t-1} \mid x_t) - \log q(x_t \mid x_{t-1})\right)$$

$$= \log p(x_T) + \sum_{t \geq 1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_t \mid x_{t-1})}$$

So,

$$\mathbb{E}\!\left[-\log p_\theta(x_0)\right] \leq \mathbb{E}_q\!\left[-\log \frac{p_\theta(x_{0:T})}{q(x_{1:T} \mid x_0)}\right] = \mathbb{E}_q\!\left[-\log p(x_T) - \sum_{t \geq 1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_t \mid x_{t-1})}\right] =: L \qquad (3)$$

---

## 11. Facts about Gaussians

**Fact A.** If $Z \sim \mathcal{N}(0, \sigma^2)$ and $c$ is a constant, then

$$cZ \sim \mathcal{N}(0,\ c^2\sigma^2)$$

**Fact B.** If $Z_1 \sim \mathcal{N}(0, \sigma_1^2)$ and $Z_2 \sim \mathcal{N}(0, \sigma_2^2)$ are independent, then

$$Z_1 + Z_2 \sim \mathcal{N}(0,\ \sigma_1^2 + \sigma_2^2)$$

**Fact C.** Any zero-mean Gaussian can be written as its own standard deviation times a standard normal. If $Y \sim \mathcal{N}(0, \sigma^2)$, define

$$\bar{\epsilon} := \frac{Y}{\sigma}$$

By Fact A, $\mathrm{Var}(\bar{\epsilon}) = \frac{1}{\sigma^2}\sigma^2 = 1$, so $\bar{\epsilon} \sim \mathcal{N}(0,1)$. So,

$$Y = \sigma \bar{\epsilon} = \sqrt{\mathrm{Var}(Y)}\ \bar{\epsilon}$$

### $\alpha_t$ and $\bar{\alpha}_t$

$$\alpha_t := 1 - \beta_t$$

→ fraction of variance or signal kept at step $t$.

$$\bar{\alpha}_t := \prod_{s=1}^{t} \alpha_s$$

→ product of all the signals kept from step 1 to step $t$. It is the fraction of the original signal that survives after step $t$.

> **Note:** $\bar{\alpha}_t = \alpha_t \cdot \bar{\alpha}_{t-1}$, because $\bar{\alpha}_t = \alpha_t \cdot \underbrace{\alpha_{t-1} \cdot \alpha_{t-2} \cdots \alpha_1}_{\bar{\alpha}_{t-1}}$

### "Merging two noise draws"

Fact B and Fact C together let us replace a sum of two random variables with a single one.

If $Y = c_1 \epsilon_1 + c_2 \epsilon_2$, then by Fact B, $Y$ is Gaussian and its variance is $c_1^2 + c_2^2$. By Fact C, $\sqrt{c_1^2 + c_2^2}\ \bar{\epsilon}$ is a Gaussian with exactly that mean and variance.

This is equality **in distribution**, written $\stackrel{d}{=}$, not equality number-for-number:

$$c_1 \epsilon_1 + c_2 \epsilon_2 \stackrel{d}{=} \sqrt{c_1^2 + c_2^2}\ \bar{\epsilon}$$

---

## 12. Equation (4) — sampling $x_t$ directly from $x_0$

$$q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1-\bar{\alpha}_t) I\right) \qquad (4)$$

### Derivation

We already have

$$q(x_t \mid x_{t-1}) := \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right) \qquad (2)$$

$$x_t = \sqrt{1-\beta_t}\, x_{t-1} + \sqrt{\beta_t}\, \epsilon$$

Now, $\alpha_t = 1 - \beta_t$ and $\beta_t = 1 - \alpha_t$. So,

$$x_t = \sqrt{\alpha_t}\, x_{t-1} + \sqrt{1-\alpha_t}\ \epsilon_t, \qquad \epsilon_t \sim \mathcal{N}(0, I)$$

Now,

$$x_{t-1} = \sqrt{\alpha_{t-1}}\, x_{t-2} + \sqrt{1-\alpha_{t-1}}\ \epsilon_{t-1}$$

So,

$$x_t = \sqrt{\alpha_t}\left(\sqrt{\alpha_{t-1}}\, x_{t-2} + \sqrt{1-\alpha_{t-1}}\ \epsilon_{t-1}\right) + \sqrt{1-\alpha_t}\ \epsilon_t$$

$$\Rightarrow \quad x_t = \sqrt{\alpha_t \alpha_{t-1}}\, x_{t-2} + \sqrt{\alpha_t}\sqrt{1-\alpha_{t-1}}\ \epsilon_{t-1} + \sqrt{1-\alpha_t}\ \epsilon_t$$

Looking at the last two terms, together they form a single random variable; call it

$$Y := \sqrt{\alpha_t}\sqrt{1-\alpha_{t-1}}\ \epsilon_{t-1} + \sqrt{1-\alpha_t}\ \epsilon_t$$

By Fact B, $Y$ is Gaussian; by Facts A and B, its variance is the sum of the two squared coefficients:

$$\mathrm{Var}(Y) = \alpha_t(1-\alpha_{t-1}) + (1-\alpha_t) = \alpha_t - \alpha_t\alpha_{t-1} + 1 - \alpha_t = 1 - \alpha_t\alpha_{t-1}$$

By Fact C, take the square root of the variance to get the standard deviation, and write

$$Y = \sqrt{1 - \alpha_t \alpha_{t-1}}\ \bar{\epsilon}, \qquad \bar{\epsilon} \sim \mathcal{N}(0, I)$$

Substituting $Y$ back in the equation of $x_t$:

$$x_t = \sqrt{\alpha_t \alpha_{t-1}}\, x_{t-2} + \sqrt{1 - \alpha_t \alpha_{t-1}}\ \bar{\epsilon}, \qquad \bar{\epsilon} \sim \mathcal{N}(0, I)$$

To be precise, this is $\stackrel{d}{=}$, equality in distribution. $\bar{\epsilon}$ is a **new** variable built from the other two, not one of them.

Now, $x_t$ can be rewritten as

$$x_t = \sqrt{\alpha_t \alpha_{t-1} \alpha_{t-2}}\, x_{t-3} + \sqrt{1 - \alpha_t \alpha_{t-1} \alpha_{t-2}}\ \bar{\epsilon}$$

Continuing all the way to $x_0$, the product becomes $\prod_{s=1}^{t} \alpha_s = \bar{\alpha}_t$:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\ \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

Thus,

$$q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1-\bar{\alpha}_t) I\right) \qquad (4)$$

---

## 13. Entropy — the average surprisal

We know that $-\log p(x)$ shows how surprised we are to see the outcome $x$. **Entropy** is the average surprisal, taken over the distribution's own outcomes:

$$H(p) = \mathbb{E}_{x \sim p}\!\left[-\log p(x)\right] = -\sum_x p(x) \log p(x)$$

If outcomes are drawn from $p$, how surprised am I on average? It measures how spread out or unpredictable $p$ is.

**Example.** A fair coin, $p = (0.5, 0.5)$, using $\log_2$:

$$H = -0.5 \log_2 0.5 - 0.5 \log_2 0.5 = 0.5 + 0.5 = 1$$

A biased coin, $p = (0.9, 0.1)$:

$$H = -0.9 \log_2 0.9 - 0.1 \log_2 0.1 = 0.137 + 0.332 = 0.469$$

The biased coin is more predictable, so lower entropy. A certain outcome, $p = (1, 0)$, gives $H = 0$ — no surprise ever.

For a continuous distribution, sums become integrals and this is called **differential entropy**.

---

## 14. Cross-entropy and KL divergence

Now suppose the truth is $q$, but I have built my model $p$ and I use $p$'s surprisal values. My average surprise is the **cross-entropy**:

$$H(q, p) = \mathbb{E}_{x \sim q}\!\left[-\log p(x)\right]$$

Outcomes come from $q$, but the surprisal is computed with $p$. If my model is wrong, I will be surprised more often than necessary.

The **Kullback–Leibler divergence** is exactly that excess — how much extra surprise I suffer by using $p$ when the truth is $q$:

$$D_{\mathrm{KL}}(q \parallel p) = H(q, p) - H(q)$$

$$D_{\mathrm{KL}}(q \parallel p) = \mathbb{E}_{x \sim q}\!\left[-\log p(x)\right] - \mathbb{E}_{x \sim q}\!\left[-\log q(x)\right]$$

$$D_{\mathrm{KL}}(q \parallel p) = \mathbb{E}_{x \sim q}\!\left[\log \frac{q(x)}{p(x)}\right]$$

(the direction is "from $q$ to $p$")

$$\Rightarrow \quad D_{\mathrm{KL}}(q \parallel p) = \int q(x) \log \frac{q(x)}{p(x)}\, dx$$

### Three properties of $D_{\mathrm{KL}}$

1. It is never negative: $D_{\mathrm{KL}}(q \parallel p) \geq 0$
2. It is zero exactly when $q = p$
3. It is not symmetric: $D_{\mathrm{KL}}(q \parallel p) \neq D_{\mathrm{KL}}(p \parallel q)$

---

## 15. Example — ranking CMIP6 models by KL divergence

I have downscaled CMIP6 projections for future temperature data. I downscaled both past data from 1981 to 2014 and projections data from 2015 to 2100. I also have observation data from PRISM.

I used 41 CMIP6 models, and computing $D_{\mathrm{KL}}(\text{obs} \parallel \text{model}_i)$ for each model gives an ordering. If

$$D_{\mathrm{KL}}(\text{obs} \parallel \text{model}_{12}) < D_{\mathrm{KL}}(\text{obs} \parallel \text{model}_{30})$$

we can say that model 12 is more distributionally faithful than model 30.

### Computing $D_{\mathrm{KL}}(\text{obs} \parallel \text{model})$

1. Pool both series and set common bin edges.
2. Count how many days fall in each bin, separately for obs and downscaled.
3. Normalize each count vector to sum to 1, so $q_i$ and $p_i$ are probabilities.

Let

- $o_i$ = number of observed days in bin $i$
- $d_i$ = number of downscaled days in bin $i$
- $B$ = number of bins (for example, 13 here)

$$q_i = \frac{o_i + \varepsilon}{\sum_{j=1}^{B}(o_j + \varepsilon)}, \qquad p_i = \frac{d_i + \varepsilon}{\sum_{j=1}^{B}(d_j + \varepsilon)}$$

Here $\varepsilon = 0.5$ is the smoothing constant. Then

$$D_{\mathrm{KL}}(q \parallel p) = \sum_{i=1}^{B} q_i \log \frac{q_i}{p_i}$$

Suppose $\sum_{j=1}^{B}(o_j + \varepsilon) = 10957.5 = \sum_{j=1}^{B}(d_j + \varepsilon)$. Then:

| bin (°C) | $o_i$ | $d_i$ | $q_i$ | $p_i$ | $q_i \log(q_i/p_i)$ |
|:---|---:|---:|---:|---:|---:|
| $< -6$ | 0 | 0 | 0.000046 | 0.000046 | 0.0 |
| $-6$ to $-2$ | 0 | 0 | 0.000046 | 0.000046 | 0.0 |
| $-2$ to $2$ | 3 | 0 | 0.000319 | 0.000046 | 0.000622 |
| $2$ to $6$ | 43 | 12 | 0.00397 | 0.00141 | 0.0041957 |
| $\vdots$ | | | | | |
| $\geq 46$ | 1 | 0 | 0.000137 | 0.000046 | 0.000150 |
| **Total** | 10950 | 10950 | 1.0 | 1.0 | **0.2285** |

So, $D_{\mathrm{KL}}(\text{obs} \parallel \text{model}) = 0.2285$.

---

## 16. $D_{\mathrm{KL}}(P \parallel Q)$ when $P$ and $Q$ are both normal

$$D_{\mathrm{KL}}(P \parallel Q) = \mathbb{E}_{x \sim P}\!\left[\log P(x) - \log Q(x)\right]$$

where $P = \mathcal{N}(\mu_1, \sigma_1^2)$ and $Q = \mathcal{N}(\mu_2, \sigma_2^2)$.

$$P(x) = \frac{1}{\sigma_1 \sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu_1)^2}{2\sigma_1^2}\right), \qquad Q(x) = \frac{1}{\sigma_2 \sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu_2)^2}{2\sigma_2^2}\right)$$

$$\log P(x) = -\log \sigma_1 - \tfrac{1}{2}\log(2\pi) - \frac{(x-\mu_1)^2}{2\sigma_1^2}$$

$$\log Q(x) = -\log \sigma_2 - \tfrac{1}{2}\log(2\pi) - \frac{(x-\mu_2)^2}{2\sigma_2^2}$$

$$\log P(x) - \log Q(x) = -\frac{(x-\mu_1)^2}{2\sigma_1^2} - \log \sigma_1 + \frac{(x-\mu_2)^2}{2\sigma_2^2} + \log \sigma_2$$

So,

$$D_{\mathrm{KL}}(P \parallel Q) = \mathbb{E}_{x \sim P}\!\left[-\frac{(x-\mu_1)^2}{2\sigma_1^2} + \frac{(x-\mu_2)^2}{2\sigma_2^2} + \log \frac{\sigma_2}{\sigma_1}\right]$$

Now,

$$\mathbb{E}_{x \sim P}\!\left[-\frac{(x-\mu_1)^2}{2\sigma_1^2}\right] = -\frac{\sigma_1^2}{2\sigma_1^2} = -\frac{1}{2}$$

And expanding the second term:

$$(x-\mu_2)^2 = \left[(x-\mu_1) + (\mu_1-\mu_2)\right]^2 = (x-\mu_1)^2 + 2(x-\mu_1)(\mu_1-\mu_2) + (\mu_1-\mu_2)^2$$

Now, term by term:

$$\mathbb{E}_{x \sim P}\!\left[(x-\mu_1)^2\right] = \sigma_1^2$$

$$\mathbb{E}_{x \sim P}\!\left[2(x-\mu_1)(\mu_1-\mu_2)\right] = 2(\mu_1-\mu_2)\,\mathbb{E}\!\left[x - \mu_1\right] = 0$$

$$\mathbb{E}\!\left[(\mu_1-\mu_2)^2\right] = (\mu_1-\mu_2)^2 \quad \text{— a constant}$$

So,

$$\mathbb{E}_{x \sim P}\!\left[\frac{(x-\mu_2)^2}{2\sigma_2^2}\right] = \frac{\sigma_1^2 + (\mu_1-\mu_2)^2}{2\sigma_2^2}$$

Thus,

$$D_{\mathrm{KL}}(P \parallel Q) = \log \frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$$

---

## 17. Equation (5) — the three-term decomposition

$$\mathbb{E}_q\!\left[\underbrace{D_{\mathrm{KL}}\!\left(q(x_T \mid x_0) \parallel p(x_T)\right)}_{L_T} + \sum_{t>1} \underbrace{D_{\mathrm{KL}}\!\left(q(x_{t-1} \mid x_t, x_0) \parallel p_\theta(x_{t-1} \mid x_t)\right)}_{L_{t-1}} - \underbrace{\log p_\theta(x_0 \mid x_1)}_{L_0}\right] \qquad (5)$$

> **What is $\mathbb{E}_q$?**
>
> $\mathbb{E}_q$ is an average over the forward process. Draw a training image $x_0 \sim q(x_0)$, then run the noising chain to get a trajectory $x_1, x_2, \ldots, x_T \sim q(x_{1:T} \mid x_0)$. Every quantity inside the bracket is then evaluated on that trajectory, and you average over many of them.

### Derivation

We have, from Eq. (3):

$$\mathbb{E}\!\left[-\log p_\theta(x_0)\right] \leq \mathbb{E}_q\!\left[-\log p(x_T) - \sum_{t \geq 1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_t \mid x_{t-1})}\right] =: L \qquad (3)$$

Here, let

$$\alpha = -\log p(x_T) - \sum_{t \geq 1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_t \mid x_{t-1})}$$

Pull the $t = 1$ term out of the sum:

$$\alpha = -\log p(x_T) - \log \frac{p_\theta(x_0 \mid x_1)}{q(x_1 \mid x_0)} - \sum_{t > 1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_t \mid x_{t-1})}$$

Now, using **Bayes' rule**:

$$q(x_t \mid x_{t-1}, x_0) = \frac{q(x_{t-1} \mid x_t, x_0)\, q(x_t \mid x_0)}{q(x_{t-1} \mid x_0)}$$

Using the Markov property, we can drop $x_0$ on the left:

$$q(x_t \mid x_{t-1}) = \frac{q(x_{t-1} \mid x_t, x_0)\, q(x_t \mid x_0)}{q(x_{t-1} \mid x_0)}$$

So,

$$L = -\log p(x_T) - \log \frac{p_\theta(x_0 \mid x_1)}{q(x_1 \mid x_0)} - \sum_{t > 1} \log \left[\frac{p_\theta(x_{t-1} \mid x_t)}{q(x_{t-1} \mid x_t, x_0)} \cdot \frac{q(x_{t-1} \mid x_0)}{q(x_t \mid x_0)}\right]$$

Now, $\sum \log(A \cdot B) = \sum (\log A + \log B)$, so

$$\sum_{t>1} \log \left[\frac{p_\theta(x_{t-1} \mid x_t)}{q(x_{t-1} \mid x_t, x_0)} \cdot \frac{q(x_{t-1} \mid x_0)}{q(x_t \mid x_0)}\right] = \sum_{t>1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_{t-1} \mid x_t, x_0)} + \sum_{t>1} \log \frac{q(x_{t-1} \mid x_0)}{q(x_t \mid x_0)}$$

### The telescoping sum

$$\sum_{t>1} \log \frac{q(x_{t-1} \mid x_0)}{q(x_t \mid x_0)} = \log \frac{q(x_1 \mid x_0)}{q(x_2 \mid x_0)} + \log \frac{q(x_2 \mid x_0)}{q(x_3 \mid x_0)} + \cdots + \log \frac{q(x_{T-1} \mid x_0)}{q(x_T \mid x_0)}$$

$$= \log \left(\frac{q(x_1 \mid x_0)}{q(x_2 \mid x_0)} \cdot \frac{q(x_2 \mid x_0)}{q(x_3 \mid x_0)} \cdot \frac{q(x_3 \mid x_0)}{q(x_4 \mid x_0)} \cdots \frac{q(x_{T-1} \mid x_0)}{q(x_T \mid x_0)}\right)$$

$$= \log \frac{q(x_1 \mid x_0)}{q(x_T \mid x_0)}$$

### Putting it together

Thus,

$$\alpha = -\log p(x_T) - \sum_{t>1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_{t-1} \mid x_t, x_0)} - \log \frac{q(x_1 \mid x_0)}{q(x_T \mid x_0)} - \log \frac{p_\theta(x_0 \mid x_1)}{q(x_1 \mid x_0)}$$

$$= -\log p(x_T) - \sum_{t>1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_{t-1} \mid x_t, x_0)} - \log q(x_1 \mid x_0) + \log q(x_T \mid x_0) - \log p_\theta(x_0 \mid x_1) + \log q(x_1 \mid x_0)$$

The two $\log q(x_1 \mid x_0)$ terms cancel, leaving

$$\alpha = \log \frac{q(x_T \mid x_0)}{p(x_T)} - \sum_{t>1} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_{t-1} \mid x_t, x_0)} - \log p_\theta(x_0 \mid x_1)$$

So,

$$L = \mathbb{E}_q\!\left[\log \frac{q(x_T \mid x_0)}{p(x_T)} + \sum_{t>1} \log \frac{q(x_{t-1} \mid x_t, x_0)}{p_\theta(x_{t-1} \mid x_t)} - \log p_\theta(x_0 \mid x_1)\right]$$

Each of the first two pieces is exactly a KL divergence, so

$$L = \mathbb{E}_q\!\left[\underbrace{D_{\mathrm{KL}}\!\left(q(x_T \mid x_0) \parallel p(x_T)\right)}_{L_T} + \sum_{t>1} \underbrace{D_{\mathrm{KL}}\!\left(q(x_{t-1} \mid x_t, x_0) \parallel p_\theta(x_{t-1} \mid x_t)\right)}_{L_{t-1}} - \underbrace{\log p_\theta(x_0 \mid x_1)}_{L_0}\right] \qquad (5)$$

---

## 18. Equations (6) and (7) — the tractable forward posterior $q(x_{t-1} \mid x_t, x_0)$

$$q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\!\left(x_{t-1};\ \tilde{\mu}_t(x_t, x_0),\ \tilde{\beta}_t I\right) \qquad (6)$$

where

$$\tilde{\mu}_t(x_t, x_0) := \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1-\bar{\alpha}_t}\, x_0 + \frac{\sqrt{\alpha_t}\,\left(1-\bar{\alpha}_{t-1}\right)}{1-\bar{\alpha}_t}\, x_t$$

and

$$\tilde{\beta}_t := \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}\, \beta_t \qquad (7)$$

### Why we need Bayes' rule here

In equation (5), we have this form $q(x_{t-1} \mid x_t, x_0)$ — but this $q$ was defined for the **forward** process, and not the backward process.

Using the Bayesian formula:

$$q(x_{t-1} \mid x_t, x_0) = \frac{q(x_t \mid x_{t-1}, x_0)\, q(x_{t-1} \mid x_0)}{q(x_t \mid x_0)}$$

By the Markov property of the forward process, once $x_{t-1}$ is known, $x_t$ does not care where the chain has been:

$$q(x_t \mid x_{t-1}, x_0) = q(x_t \mid x_{t-1})$$

So,

$$q(x_{t-1} \mid x_t, x_0) = \frac{q(x_t \mid x_{t-1})\, q(x_{t-1} \mid x_0)}{q(x_t \mid x_0)}$$

### The three Gaussians on the right-hand side

From equation (2), with $\beta_t = 1 - \alpha_t$:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{\alpha_t}\, x_{t-1},\ \beta_t I\right)$$

From Eq. (4) applied at step $t-1$:

$$q(x_{t-1} \mid x_0) = \mathcal{N}\!\left(x_{t-1};\ \sqrt{\bar{\alpha}_{t-1}}\, x_0,\ \left(1-\bar{\alpha}_{t-1}\right) I\right)$$

From Eq. (4) applied at step $t$:

$$q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ \left(1-\bar{\alpha}_t\right) I\right)$$

For a normal distribution with mean $\mu$ and variance $\sigma^2$:

$$q(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{1}{2}\, \frac{(x-\mu)^2}{\sigma^2}\right)$$

So, keeping only the exponential parts:

$$q(x_t \mid x_{t-1}) \propto \exp\!\left[-\frac{1}{2}\, \frac{\left(x_t - \sqrt{\alpha_t}\, x_{t-1}\right)^2}{\beta_t}\right]$$

$$q(x_{t-1} \mid x_0) \propto \exp\!\left[-\frac{1}{2}\, \frac{\left(x_{t-1} - \sqrt{\bar{\alpha}_{t-1}}\, x_0\right)^2}{1-\bar{\alpha}_{t-1}}\right]$$

$$q(x_t \mid x_0) \propto \exp\!\left[-\frac{1}{2}\, \frac{\left(x_t - \sqrt{\bar{\alpha}_t}\, x_0\right)^2}{1-\bar{\alpha}_t}\right]$$

### Multiplying the three exponents together

Now,

$$q(x_{t-1} \mid x_t, x_0) = \frac{q(x_t \mid x_{t-1})\, q(x_{t-1} \mid x_0)}{q(x_t \mid x_0)}$$

$$\propto \exp\!\left(-\frac{1}{2}\left[\frac{\left(x_t-\sqrt{\alpha_t}\, x_{t-1}\right)^2}{\beta_t} + \frac{\left(x_{t-1}-\sqrt{\bar{\alpha}_{t-1}}\, x_0\right)^2}{1-\bar{\alpha}_{t-1}} - \frac{\left(x_t-\sqrt{\bar{\alpha}_t}\, x_0\right)^2}{1-\bar{\alpha}_t}\right]\right)$$

Keeping the normalizing constants too:

$$q(x_{t-1} \mid x_t, x_0) = \frac{1}{\sqrt{2\pi\beta_t}} \cdot \frac{1}{\sqrt{2\pi\left(1-\bar{\alpha}_{t-1}\right)}} \Bigg/ \left(\frac{1}{\sqrt{2\pi\left(1-\bar{\alpha}_t\right)}}\right) \times \exp\!\left(-\frac{1}{2}\left[\cdots\right]\right)$$

The three $\sqrt{2\pi}$ factors collapse to one, leaving

$$q(x_{t-1} \mid x_t, x_0) = \frac{1}{\sqrt{2\pi}} \sqrt{\frac{1-\bar{\alpha}_t}{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}}\ \exp\!\left(-\frac{1}{2}\left[\frac{\left(x_t-\sqrt{\alpha_t}\, x_{t-1}\right)^2}{\beta_t} + \frac{\left(x_{t-1}-\sqrt{\bar{\alpha}_{t-1}}\, x_0\right)^2}{1-\bar{\alpha}_{t-1}} - \frac{\left(x_t-\sqrt{\bar{\alpha}_t}\, x_0\right)^2}{1-\bar{\alpha}_t}\right]\right)$$

### Naming the three quadratic pieces

Let

$$s_1 = \frac{\left(x_t - \sqrt{\alpha_t}\, x_{t-1}\right)^2}{\beta_t}, \qquad s_2 = \frac{\left(x_{t-1} - \sqrt{\bar{\alpha}_{t-1}}\, x_0\right)^2}{1-\bar{\alpha}_{t-1}}, \qquad s_3 = \frac{\left(x_t - \sqrt{\bar{\alpha}_t}\, x_0\right)^2}{1-\bar{\alpha}_t}$$

So,

$$q(x_{t-1} \mid x_t, x_0) = \frac{1}{\sqrt{2\pi}} \sqrt{\frac{1-\bar{\alpha}_t}{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}}\ \exp\!\left(\frac{s_3}{2}\right) \exp\!\left(-\frac{1}{2}\left(s_1+s_2\right)\right)$$

$$q(x_{t-1} \mid x_t, x_0) \propto \exp\!\left(\frac{s_3}{2}\right) \exp\!\left(-\frac{1}{2}\left(s_1+s_2\right)\right)$$

### Expanding $s_1$ and $s_2$

From $s_1$:

$$\frac{\left(x_t - \sqrt{\alpha_t}\, x_{t-1}\right)^2}{\beta_t} = \underbrace{\frac{\alpha_t}{\beta_t}\, x_{t-1}^2 - \frac{2\sqrt{\alpha_t}}{\beta_t}\, x_t\, x_{t-1}}_{\text{has } x_{t-1} \text{ terms}} + \underbrace{\frac{x_t^2}{\beta_t}}_{\text{constant}}$$

From $s_2$:

$$\frac{\left(x_{t-1} - \sqrt{\bar{\alpha}_{t-1}}\, x_0\right)^2}{1-\bar{\alpha}_{t-1}} = \underbrace{\frac{x_{t-1}^2}{1-\bar{\alpha}_{t-1}} - \frac{2\sqrt{\bar{\alpha}_{t-1}}}{1-\bar{\alpha}_{t-1}}\, x_0\, x_{t-1}}_{\text{has } x_{t-1}} + \underbrace{\frac{\bar{\alpha}_{t-1}\, x_0^2}{1-\bar{\alpha}_{t-1}}}_{\text{constant}}$$

Just consider $s_1 + s_2$:

$$s_1 + s_2 = \left(\frac{\alpha_t}{\beta_t} + \frac{1}{1-\bar{\alpha}_{t-1}}\right) x_{t-1}^2 - 2\left(\frac{\sqrt{\alpha_t}}{\beta_t}\, x_t + \frac{\sqrt{\bar{\alpha}_{t-1}}}{1-\bar{\alpha}_{t-1}}\, x_0\right) x_{t-1} + \frac{x_t^2}{\beta_t} + \frac{\bar{\alpha}_{t-1}\, x_0^2}{1-\bar{\alpha}_{t-1}}$$

Note that $s_3$ has no $x_{t-1}$ in it at all — it is a constant as far as $x_{t-1}$ is concerned.

### Completing the square in $x_{t-1}$

Let

$$A = \frac{\alpha_t}{\beta_t} + \frac{1}{1-\bar{\alpha}_{t-1}} \qquad \text{and} \qquad b = \frac{\sqrt{\alpha_t}}{\beta_t}\, x_t + \frac{\sqrt{\bar{\alpha}_{t-1}}}{1-\bar{\alpha}_{t-1}}\, x_0$$

So,

$$s_1 + s_2 = A\, x_{t-1}^2 - 2b\, x_{t-1} + \frac{x_t^2}{\beta_t} + \frac{\bar{\alpha}_{t-1}\, x_0^2}{1-\bar{\alpha}_{t-1}}$$

$$\Rightarrow \quad s_1 + s_2 = A\left(x_{t-1} - \frac{b}{A}\right)^2 + \underbrace{\frac{x_t^2}{\beta_t} + \frac{\bar{\alpha}_{t-1}\, x_0^2}{1-\bar{\alpha}_{t-1}} - \frac{b^2}{A}}_{R}$$

$$\Rightarrow \quad s_1 + s_2 = A\left(x_{t-1} - \frac{b}{A}\right)^2 + R$$

So,

$$\exp\!\left(-\frac{1}{2}\left(s_1+s_2\right)\right) = \exp\!\left(-\frac{A}{2}\left(x_{t-1}-\frac{b}{A}\right)^2\right) \cdot \exp\!\left(-\frac{R}{2}\right)$$

So,

$$q(x_{t-1} \mid x_t, x_0) = \frac{1}{\sqrt{2\pi}} \sqrt{\frac{1-\bar{\alpha}_t}{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}}\ \exp\!\left(\frac{s_3}{2}\right) \exp\!\left(-\frac{R}{2}\right) \exp\!\left(-\frac{A}{2}\left(x_{t-1}-\frac{b}{A}\right)^2\right)$$

### Showing that $R = s_3$, so the two leftover factors cancel

Shorthand for this part: let $\alpha := \alpha_t$, $u := \bar{\alpha}_{t-1}$, $\beta := \beta_t = 1-\alpha$, so that $\bar{\alpha}_t = \alpha u$.

$$R = \frac{x_t^2}{\beta} + \frac{u\, x_0^2}{1-u} - \frac{b^2}{A}$$

with

$$b = \frac{\sqrt{\alpha}}{\beta}\, x_t + \frac{\sqrt{u}}{1-u}\, x_0, \qquad A = \frac{\alpha}{\beta} + \frac{1}{1-u} = \frac{\alpha - \alpha u + \beta}{\beta(1-u)}$$

$$\Rightarrow \quad A = \frac{\alpha + \beta - \alpha u}{\beta(1-u)} = \frac{1-\alpha u}{\beta(1-u)}$$

$$\frac{1}{A} = \frac{\beta(1-u)}{1-\alpha u}$$

$$b^2 = \frac{\alpha}{\beta^2}\, x_t^2 + \frac{2\sqrt{\alpha u}}{\beta(1-u)}\, x_t x_0 + \frac{u}{(1-u)^2}\, x_0^2$$

So,

$$\frac{b^2}{A} = \frac{\alpha(1-u)}{\beta(1-\alpha u)}\, x_t^2 + \frac{2\sqrt{\alpha u}}{1-\alpha u}\, x_t x_0 + \frac{u\beta}{(1-u)(1-\alpha u)}\, x_0^2$$

#### Now collect $R$ coefficient by coefficient

**Collect the $x_t^2$ coefficient:**

$$\frac{1}{\beta} - \frac{\alpha(1-u)}{\beta(1-\alpha u)} = \frac{(1-\alpha u) - \alpha(1-u)}{\beta(1-\alpha u)} = \frac{1 - \alpha u - \alpha + \alpha u}{\beta(1-\alpha u)}$$

$$= \frac{1-\alpha}{\beta(1-\alpha u)} = \frac{\beta}{\beta(1-\alpha u)} = \frac{1}{1-\alpha u}$$

**Collect the $x_t x_0$ terms:**

$$-\frac{2\sqrt{\alpha u}}{1-\alpha u}$$

**Collect the $x_0^2$ terms:**

$$\frac{u}{1-u} - \frac{u\beta}{(1-u)(1-\alpha u)} = \frac{u\left[(1-\alpha u) - \beta\right]}{(1-u)(1-\alpha u)} = \frac{u\left[1 - \alpha u - 1 + \alpha\right]}{(1-u)(1-\alpha u)}$$

$$= \frac{u\alpha(1-u)}{(1-u)(1-\alpha u)} = \frac{\alpha u}{1-\alpha u}$$

So,

$$R = \frac{1}{1-\alpha u}\, x_t^2 - \frac{2}{1-\alpha u}\sqrt{\alpha u}\, x_t x_0 + \frac{\alpha u}{1-\alpha u}\, x_0^2$$

$$= \frac{1}{1-\alpha u}\left(x_t^2 - 2 x_t \sqrt{\alpha u}\, x_0 + \alpha u\, x_0^2\right) = \frac{\left(x_t - \sqrt{\alpha u}\, x_0\right)^2}{1-\alpha u}$$

$$\Rightarrow \quad R = \frac{\left(x_t - \sqrt{\bar{\alpha}_t}\, x_0\right)^2}{1-\bar{\alpha}_t}$$

As $s_3 = R$:

$$\exp\!\left(\frac{s_3}{2}\right) \times \exp\!\left(-\frac{R}{2}\right) = 1$$

### Reading off the mean and the variance

So,

$$q(x_{t-1} \mid x_t, x_0) = \frac{1}{\sqrt{2\pi}} \sqrt{\frac{1-\bar{\alpha}_t}{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}}\ \exp\!\left(-\frac{A}{2}\left(x_{t-1}-\frac{b}{A}\right)^2\right)$$

This is exactly a Gaussian in $x_{t-1}$. Let $\mu = \dfrac{b}{A}$ and $\sigma^2 = \dfrac{1}{A}$.

$$A = \frac{\alpha_t}{\beta_t} + \frac{1}{1-\bar{\alpha}_{t-1}} = \frac{\alpha_t\left(1-\bar{\alpha}_{t-1}\right) + \beta_t}{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}$$

$$= \frac{\alpha_t - \alpha_t \bar{\alpha}_{t-1} + \beta_t}{\beta_t\left(1-\bar{\alpha}_{t-1}\right)} = \frac{1-\bar{\alpha}_t}{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}$$

(using $\alpha_t + \beta_t = 1$ and $\alpha_t \bar{\alpha}_{t-1} = \bar{\alpha}_t$)

So,

$$\sigma^2 = \frac{1}{A} = \frac{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}{1-\bar{\alpha}_t} =: \tilde{\beta}_t$$

So,

$$\frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}\, \beta_t = \tilde{\beta}_t \qquad (7)$$

And for the mean:

$$\mu = \frac{b}{A} = \frac{\sqrt{\alpha_t}}{\beta_t} \cdot \frac{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}{1-\bar{\alpha}_t}\, x_t + \frac{\sqrt{\bar{\alpha}_{t-1}}}{1-\bar{\alpha}_{t-1}} \cdot \frac{\beta_t\left(1-\bar{\alpha}_{t-1}\right)}{1-\bar{\alpha}_t}\, x_0$$

$$\Rightarrow \quad \mu = \frac{\sqrt{\alpha_t}\left(1-\bar{\alpha}_{t-1}\right)}{1-\bar{\alpha}_t}\, x_t + \frac{\sqrt{\bar{\alpha}_{t-1}}\, \beta_t}{1-\bar{\alpha}_t}\, x_0$$

$$\Rightarrow \quad \mu = \frac{\sqrt{\bar{\alpha}_{t-1}}\, \beta_t}{1-\bar{\alpha}_t}\, x_0 + \frac{\sqrt{\alpha_t}\left(1-\bar{\alpha}_{t-1}\right)}{1-\bar{\alpha}_t}\, x_t =: \tilde{\mu}_t(x_t, x_0) \qquad (7)$$

Thus,

$$q(x_{t-1} \mid x_t, x_0) = \frac{1}{\sqrt{2\pi\tilde{\beta}_t}} \exp\!\left(-\frac{\left(x_{t-1} - \tilde{\mu}_t\right)^2}{2\tilde{\beta}_t}\right) = \mathcal{N}\!\left(x_{t-1};\ \tilde{\mu}_t(x_t, x_0),\ \tilde{\beta}_t I\right) \qquad (6)$$

This is the result the paper states in Eqs. (6) and (7): the forward posterior is itself Gaussian, with a mean that is a fixed linear blend of $x_t$ and $x_0$, and a variance that depends only on the noise schedule. Because it is Gaussian, every KL term $L_{t-1}$ in Eq. (5) is a KL between two Gaussians and can be written in closed form — no Monte Carlo estimate needed.

---

## 19. The Rao–Blackwell theorem

If we have an unbiased estimator $\hat{G}$ of some quantity, and you condition on some information $Y$ and take the expectation, then the conditioned estimator

$$\hat{G}' = \mathbb{E}\!\left[\hat{G} \mid Y\right]$$

is still unbiased, and its variance is never larger:

$$\mathrm{Var}\!\left(\hat{G}'\right) \leq \mathrm{Var}\!\left(\hat{G}\right)$$

The reason is the **law of total variance**:

$$\mathrm{Var}\!\left(\hat{G}\right) = \mathbb{E}\!\left[\mathrm{Var}\!\left(\hat{G} \mid Y\right)\right] + \mathrm{Var}\!\left(\mathbb{E}\!\left[\hat{G} \mid Y\right]\right)$$

> The first two terms of equation (5) are **Rao-Blackwellized**.

The second term on the right is exactly $\mathrm{Var}(\hat{G}')$, and the first term is an average of variances, so it is $\geq 0$. Hence $\mathrm{Var}(\hat{G}') \leq \mathrm{Var}(\hat{G})$: conditioning and averaging can only remove noise, never add it.

That is what the closed-form Gaussian KLs buy us in Eq. (5). A naive estimator would sample $x_{t-1}$ and score $\log \frac{q(x_{t-1} \mid x_t, x_0)}{p_\theta(x_{t-1} \mid x_t)}$ at that single draw. Instead, $x_{t-1}$ is integrated out analytically — the KL between two Gaussians is the conditional expectation of that log-ratio given $(x_t, x_0)$ — so the sampling noise from $x_{t-1}$ disappears from the gradient estimate entirely.

---

*End of notes — coverage runs through Eqs. (6) and (7), i.e. the whole of Section 2 of the paper, stopping just before Section 3. Next: Section 3.2, the $\epsilon$-parameterization of $\mu_\theta$, Eqs. (8)–(11), and the simplified objective $L_{\text{simple}}$ in Eq. (14).*


# DDPM Study Notes — Section 3.2, Equations (8)–(10)

*Continuation of the previous notes (which ended at Eq. (7) and the Rao–Blackwell theorem). This part covers Ho, Jain & Abbeel (2020) from the start of Section 3 through the end of Eq. (10). Everything already derived earlier — $\alpha_t$, $\bar{\alpha}_t$, $\tilde{\mu}_t$, $\tilde{\beta}_t$, Eq. (4), Eq. (5), Eq. (7), the Gaussian KL formula — is used here without re-deriving it.*

---

## 20. Section 3 — what design choices are still open

Equation (5) told us *what* to minimize:

$$L = \mathbb{E}_q\!\left[\underbrace{D_{\mathrm{KL}}\!\left(q(x_T \mid x_0) \parallel p(x_T)\right)}_{L_T} + \sum_{t>1} \underbrace{D_{\mathrm{KL}}\!\left(q(x_{t-1} \mid x_t, x_0) \parallel p_\theta(x_{t-1} \mid x_t)\right)}_{L_{t-1}} - \underbrace{\log p_\theta(x_0 \mid x_1)}_{L_0}\right]$$

But it did not tell us *how* to build $p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\!\left(x_{t-1};\ \mu_\theta(x_t, t),\ \Sigma_\theta(x_t, t)\right)$. Two things are still free to choose:

1. the covariance $\Sigma_\theta(x_t, t)$ of the reverse step, and
2. the mean $\mu_\theta(x_t, t)$ of the reverse step — i.e., what exactly the neural network should output.

Section 3 of the paper walks through the terms of Eq. (5) one at a time and fixes these choices. Section 3.1 handles $L_T$, Section 3.2 handles $L_{1:T-1}$ (the middle KL terms, for $1 < t \leq T$).

### Section 3.1 in one line — $L_T$ is a constant

$$L_T = D_{\mathrm{KL}}\!\left(q(x_T \mid x_0) \parallel p(x_T)\right)$$

Both distributions here contain **no learnable parameter** $\theta$: the forward process $q$ has its $\beta_t$ fixed to the linear schedule (not learned), and the prior $p(x_T) = \mathcal{N}(0, I)$ is fixed by definition. A term with no $\theta$ in it cannot change during training — its gradient with respect to $\theta$ is zero. So $L_T$ is a constant and we simply drop it. Nothing to train there.

---

## 21. Choosing $\Sigma_\theta$ — "untrained time dependent constants"

The paper's first decision in Section 3.2:

$$\Sigma_\theta(x_t, t) = \sigma_t^2 I$$

Three pieces of jargon are packed into this line.

**"Untrained"** means the network does *not* output the variance. $\sigma_t^2$ is a fixed number we write down by hand before training starts. (The paper later shows, in Table 2, that letting the network learn the variance made training unstable — so this simple choice is also the empirically better one.)

**"Time dependent"** means the number is allowed to be different at each step $t$: $\sigma_1^2, \sigma_2^2, \ldots, \sigma_T^2$ — a lookup table indexed by $t$, not a single number.

**The $I$** (identity matrix) makes the covariance **isotropic**: every coordinate (every pixel value) gets the *same* variance $\sigma_t^2$, and there is no correlation between coordinates. "Iso" = same, "tropic" = direction: the noise looks the same in every direction. So instead of learning a full $3072 \times 3072$ covariance matrix per step for CIFAR-10, we store one scalar per step.

### The two candidate values of $\sigma_t^2$

The paper says two choices work equally well in practice:

$$\sigma_t^2 = \beta_t \qquad \text{or} \qquad \sigma_t^2 = \tilde{\beta}_t = \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}\,\beta_t$$

We already met $\tilde{\beta}_t$ in Eq. (7): it is the variance of the true forward posterior $q(x_{t-1} \mid x_t, x_0)$.

Note that $\tilde{\beta}_t \leq \beta_t$ always, because $\bar{\alpha}_{t-1} \geq \bar{\alpha}_t$ (the product $\bar{\alpha}$ only shrinks as $t$ grows), so the ratio $\frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t} \leq 1$.

**Concrete numbers** (linear schedule, $\beta_1 = 10^{-4}$, $\beta_T = 0.02$, $T = 1000$):

At $t = 2$: $\beta_2 = 1.1992 \times 10^{-4}$, $\bar{\alpha}_1 = 0.9999$, $\bar{\alpha}_2 = 0.99978$, so

$$\tilde{\beta}_2 = \frac{1 - 0.9999}{1 - 0.99978} \times 1.1992 \times 10^{-4} = 0.4547 \times \beta_2 = 5.453 \times 10^{-5}$$

At $t = 2$ the two candidates differ by more than a factor of 2. But at $t = 1000$, both $\bar{\alpha}_{999}$ and $\bar{\alpha}_{1000}$ are tiny ($\approx 4 \times 10^{-5}$), the ratio is $0.9999992$, and $\tilde{\beta}_t \approx \beta_t$. So the two choices only disagree at the very early (low-noise) steps — which is why they give similar results in practice.

---

## 22. Why $\beta_t$ and $\tilde{\beta}_t$ are the two *extremes*

The paper says: "$\sigma_t^2 = \beta_t$ is optimal for $x_0 \sim \mathcal{N}(0, I)$, and $\sigma_t^2 = \tilde{\beta}_t$ is optimal for $x_0$ deterministically set to one point."

The idea: the *ideal* reverse variance is the variance of the *true* reverse conditional $q(x_{t-1} \mid x_t)$ — how spread out $x_{t-1}$ really is once you know $x_t$. But that spread depends on what the data distribution $q(x_0)$ looks like, which we do not know. So the paper asks: what does the ideal variance become in the two most extreme imaginable cases of $q(x_0)$?

**Extreme 1 — the data is maximally spread out:** $x_0 \sim \mathcal{N}(0, I)$, i.e., the "dataset" is pure standard Gaussian noise; knowing you are looking at "the data" tells you almost nothing.

We can compute the true reverse variance exactly here (per coordinate). If $x_{t-1} \sim \mathcal{N}(0, 1)$ (unit variance is preserved at every step, as shown back in Section 5 of these notes) and

$$x_t = \sqrt{\alpha_t}\, x_{t-1} + \sqrt{\beta_t}\, \epsilon$$

then $(x_{t-1}, x_t)$ are jointly Gaussian, with $\mathrm{Var}(x_{t-1}) = 1$, $\mathrm{Var}(x_t) = \alpha_t + \beta_t = 1$, and covariance

$$\mathrm{Cov}(x_{t-1}, x_t) = \mathrm{Cov}\!\left(x_{t-1},\ \sqrt{\alpha_t}\, x_{t-1}\right) = \sqrt{\alpha_t}$$

For jointly Gaussian variables, the standard conditioning formula gives

$$\mathrm{Var}(x_{t-1} \mid x_t) = \mathrm{Var}(x_{t-1}) - \frac{\mathrm{Cov}(x_{t-1}, x_t)^2}{\mathrm{Var}(x_t)} = 1 - \frac{\alpha_t}{1} = 1 - \alpha_t = \beta_t$$

So for maximally spread data, the ideal reverse variance is exactly $\beta_t$.

**Extreme 2 — the data is maximally concentrated:** $x_0$ is one fixed point (a delta; the "dataset" contains a single image, known with certainty). Then knowing $x_t$ and knowing the (only possible) $x_0$ are the same thing, so the true reverse conditional is $q(x_{t-1} \mid x_t, x_0)$ — and we derived in Section 18 that its variance is $\tilde{\beta}_t$.

Every real dataset sits somewhere between "one point" and "pure noise", so the ideal $\sigma_t^2$ sits somewhere between $\tilde{\beta}_t$ and $\beta_t$. Since we just saw the two ends are numerically close anyway, picking either endpoint is fine.

### "Upper and lower bounds on reverse process entropy"

**Entropy** (Section 13 of these notes) is average surprisal — a measure of spread. For a Gaussian $\mathcal{N}(\mu, \sigma^2)$ the entropy is

$$H = \tfrac{1}{2}\log\!\left(2\pi e\, \sigma^2\right)$$

which only grows when $\sigma^2$ grows: more variance $\Rightarrow$ more spread $\Rightarrow$ more entropy. So saying "$\tilde{\beta}_t$ and $\beta_t$ are lower and upper bounds on the variance of the true reverse step" and saying "they are lower and upper bounds on the **reverse process entropy**" are the same statement. Concentrated data (delta) $\Rightarrow$ least uncertain reverse step $\Rightarrow$ lowest entropy $\Rightarrow$ $\tilde{\beta}_t$. Spread data ($\mathcal{N}(0,I)$) $\Rightarrow$ most uncertain reverse step $\Rightarrow$ highest entropy $\Rightarrow$ $\beta_t$.

**"Coordinatewise unit variance"** just means: each individual coordinate of the data (each pixel value) has variance 1 — which our normalized data approximately satisfies. The bound argument above assumed $\mathrm{Var}(x_0) = 1$ per coordinate, which is where this condition comes from.

---

## 23. New notation — the squared norm $\|v\|^2$

Equation (8) is written in vector form, and it uses a symbol we have not defined yet. For a vector $v = (v_1, v_2, \ldots, v_D)$ with $D$ coordinates,

$$\|v\|^2 = v_1^2 + v_2^2 + \cdots + v_D^2 = \sum_{i=1}^{D} v_i^2$$

Read $\|v\|$ as "the norm of $v$" — the ordinary Euclidean length of the vector. So $\|a - b\|^2$ is the sum, over all $D$ coordinates, of the squared coordinatewise differences: the total squared error between two vectors. For CIFAR-10, $D = 3072$, and $\|\tilde{\mu}_t - \mu_\theta\|^2$ adds up the squared mean-mismatch over all 3072 pixel values.

This means: any result we derive per coordinate (scalar) turns into the vector version by summing over coordinates — squared differences $(a_i - b_i)^2$ sum into $\|a - b\|^2$, and any constant sums into $D$ copies of itself, i.e., another constant.

---

## 24. Equation (8) — $L_{t-1}$ becomes a weighted mean-squared error

Now the second decision: how to represent the mean $\mu_\theta(x_t, t)$. The paper motivates its choice by simplifying what $L_{t-1}$ actually is, once $\Sigma_\theta = \sigma_t^2 I$ is fixed.

$L_{t-1}$ is a KL divergence between two Gaussians:

$$L_{t-1} = \mathbb{E}_q\!\left[D_{\mathrm{KL}}\!\left(q(x_{t-1} \mid x_t, x_0) \parallel p_\theta(x_{t-1} \mid x_t)\right)\right]$$

with (per coordinate)

$$q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\!\left(x_{t-1};\ \tilde{\mu}_t,\ \tilde{\beta}_t\right), \qquad p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\!\left(x_{t-1};\ \mu_\theta,\ \sigma_t^2\right)$$

We already derived the closed-form KL between two normals in Section 16:

$$D_{\mathrm{KL}}(P \parallel Q) = \log \frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$$

Now plug in $\mu_1 = \tilde{\mu}_t$, $\sigma_1^2 = \tilde{\beta}_t$, $\mu_2 = \mu_\theta$, $\sigma_2^2 = \sigma_t^2$:

$$D_{\mathrm{KL}} = \log \frac{\sigma_t}{\sqrt{\tilde{\beta}_t}} + \frac{\tilde{\beta}_t}{2\sigma_t^2} - \frac{1}{2} + \frac{\left(\tilde{\mu}_t - \mu_\theta\right)^2}{2\sigma_t^2}$$

Look at each term and ask: does it contain $\theta$?

- $\log \frac{\sigma_t}{\sqrt{\tilde{\beta}_t}}$ — only schedule quantities. No $\theta$.
- $\frac{\tilde{\beta}_t}{2\sigma_t^2}$ — only schedule quantities. No $\theta$.
- $-\frac{1}{2}$ — a number. No $\theta$.
- $\frac{(\tilde{\mu}_t - \mu_\theta)^2}{2\sigma_t^2}$ — contains $\mu_\theta$. **This is the only term the network can influence.**

Everything without $\theta$ has zero gradient and cannot affect training, so we lump all of it into one constant, called $C$. Summing over the $D$ coordinates (Section 23: squared differences become the squared norm; constants stay constants), we get exactly the paper's Eq. (8):

$$L_{t-1} = \mathbb{E}_q\!\left[\frac{1}{2\sigma_t^2}\left\|\tilde{\mu}_t(x_t, x_0) - \mu_\theta(x_t, t)\right\|^2\right] + C \qquad (8)$$

> **"where $C$ is a constant that does not depend on $\theta$"** — this is a standard move. In an optimization, any additive term with no learnable parameter inside is invisible to the optimizer; we keep the symbol $C$ only so the equation stays an equality rather than becoming approximate.

### What Eq. (8) is telling us

$L_{t-1}$ is just a **mean-squared error between two means**, weighted by $\frac{1}{2\sigma_t^2}$: the network's mean $\mu_\theta$ is being pulled toward the true forward posterior mean $\tilde{\mu}_t$, which we can compute exactly from Eq. (7) because during training we know both $x_t$ and $x_0$.

**"Parameterization"** — this word will appear constantly from here on. A parameterization is a choice of *what the network's raw output represents*. The same distribution $p_\theta(x_{t-1} \mid x_t)$ can be specified by a network that outputs $\mu_\theta$ directly, or by a network that outputs something else from which $\mu_\theta$ is then computed by a fixed formula. These are different parameterizations of the same object.

So the **most straightforward parameterization**, as the paper says, is: make the network output its guess of $\tilde{\mu}_t$ directly, and train with the MSE in Eq. (8). That would work. But the paper now massages Eq. (8) into a form that suggests a better idea.

---

## 25. Rewriting $x_t$ as a function of $(x_0, \epsilon)$

Recall Eq. (4): $q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1 - \bar{\alpha}_t) I\right)$. By Fact C (a Gaussian is its mean plus its standard deviation times standard noise), sampling from it is the same as computing

$$x_t(x_0, \epsilon) = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

**The new notation $x_t(x_0, \epsilon)$** — writing $x_t$ with arguments in parentheses emphasizes that $x_t$ is no longer treated as a random variable we sample "somehow", but as a **deterministic function** of two inputs: the clean image $x_0$ and the noise draw $\epsilon$. Feed in the same $(x_0, \epsilon)$, get the same $x_t$, every time. All the randomness has been moved into the two inputs. (This is the reparameterization trick again, now used inside the loss.)

### Why the expectation changes from $\mathbb{E}_q$ to $\mathbb{E}_{x_0, \epsilon}$

In Eq. (8) the average $\mathbb{E}_q$ ran over forward-process trajectories. But everything inside the bracket of Eq. (8) depends on the trajectory only through the pair $(x_t, x_0)$ — and we just expressed $x_t$ as a function of $(x_0, \epsilon)$. So instead of averaging over trajectories, we can equivalently average over the two primitive sources of randomness:

$$\mathbb{E}_{x_0, \epsilon}\!\left[\ \cdot\ \right] \quad = \quad \text{draw } x_0 \sim q(x_0), \text{ draw } \epsilon \sim \mathcal{N}(0, I), \text{ evaluate the bracket, average}$$

The subscripts on $\mathbb{E}$ always answer the question "average over what?" — here, over the data and over the noise. This is also exactly what a training loop does: pick a training image, pick a noise tensor, compute the loss.

### Solving the same equation for $x_0$

The function $x_t(x_0, \epsilon)$ can be inverted for $x_0$:

$$\sqrt{\bar{\alpha}_t}\, x_0 = x_t(x_0, \epsilon) - \sqrt{1 - \bar{\alpha}_t}\, \epsilon$$

$$\Rightarrow \quad x_0 = \frac{1}{\sqrt{\bar{\alpha}_t}}\left(x_t(x_0, \epsilon) - \sqrt{1 - \bar{\alpha}_t}\, \epsilon\right)$$

Read this as: *if you know the noisy image and you know exactly which noise was added, you can reconstruct the clean image by simple algebra.* During training we do know $\epsilon$ (we drew it ourselves), so this substitution is legal inside the training loss.

---

## 26. Equation (9) — substituting $x_0$ away

Equation (9) is nothing more than Eq. (8) with two mechanical substitutions:

1. the expectation $\mathbb{E}_q$ rewritten as $\mathbb{E}_{x_0, \epsilon}$ (Section 25), and
2. the second argument of $\tilde{\mu}_t(x_t, x_0)$ — the $x_0$ slot — replaced by the reconstruction formula $\frac{1}{\sqrt{\bar{\alpha}_t}}\left(x_t(x_0, \epsilon) - \sqrt{1-\bar{\alpha}_t}\, \epsilon\right)$.

That gives the paper's Eq. (9):

$$L_{t-1} - C = \mathbb{E}_{x_0, \epsilon}\!\left[\frac{1}{2\sigma_t^2}\left\|\tilde{\mu}_t\!\left(x_t(x_0,\epsilon),\ \frac{1}{\sqrt{\bar{\alpha}_t}}\left(x_t(x_0,\epsilon) - \sqrt{1-\bar{\alpha}_t}\,\epsilon\right)\right) - \mu_\theta\!\left(x_t(x_0,\epsilon),\ t\right)\right\|^2\right] \qquad (9)$$

Note the constant $C$ has simply been moved to the left-hand side ($L_{t-1} - C$ instead of $L_{t-1} = \ldots + C$); nothing changed.

Why do this at all? Because the target $\tilde{\mu}_t$ originally needed $x_0$ as an input, and the whole point of the reverse process is that at *sampling* time we will not have $x_0$. Rewriting the target in terms of $(x_t, \epsilon)$ instead of $(x_t, x_0)$ is the first step toward a network that predicts the *noise* $\epsilon$ — which is where Eq. (11) is headed.

---

## 27. Equation (10) — simplifying $\tilde{\mu}_t$ after the substitution

Now we carry out the substitution inside $\tilde{\mu}_t$ explicitly and watch it collapse. To keep the algebra light, write $x_t$ for $x_t(x_0, \epsilon)$ throughout.

Start from Eq. (7):

$$\tilde{\mu}_t(x_t, x_0) = \frac{\sqrt{\bar{\alpha}_{t-1}}\, \beta_t}{1-\bar{\alpha}_t}\, x_0 + \frac{\sqrt{\alpha_t}\left(1-\bar{\alpha}_{t-1}\right)}{1-\bar{\alpha}_t}\, x_t$$

Substitute $x_0 = \frac{1}{\sqrt{\bar{\alpha}_t}}\left(x_t - \sqrt{1-\bar{\alpha}_t}\, \epsilon\right)$ into the first term:

$$\frac{\sqrt{\bar{\alpha}_{t-1}}\, \beta_t}{1-\bar{\alpha}_t} \cdot \frac{1}{\sqrt{\bar{\alpha}_t}}\left(x_t - \sqrt{1-\bar{\alpha}_t}\, \epsilon\right)$$

Simplify the ratio of the two $\bar{\alpha}$ square roots using $\bar{\alpha}_t = \alpha_t\, \bar{\alpha}_{t-1}$:

$$\frac{\sqrt{\bar{\alpha}_{t-1}}}{\sqrt{\bar{\alpha}_t}} = \frac{\sqrt{\bar{\alpha}_{t-1}}}{\sqrt{\alpha_t}\sqrt{\bar{\alpha}_{t-1}}} = \frac{1}{\sqrt{\alpha_t}}$$

So the first term becomes

$$\frac{\beta_t}{\sqrt{\alpha_t}\left(1-\bar{\alpha}_t\right)}\, x_t - \frac{\beta_t \sqrt{1-\bar{\alpha}_t}}{\sqrt{\alpha_t}\left(1-\bar{\alpha}_t\right)}\, \epsilon$$

And since $\frac{\sqrt{1-\bar{\alpha}_t}}{1-\bar{\alpha}_t} = \frac{1}{\sqrt{1-\bar{\alpha}_t}}$, the $\epsilon$ piece is

$$-\frac{\beta_t}{\sqrt{\alpha_t}\, \sqrt{1-\bar{\alpha}_t}}\, \epsilon$$

**Now collect the coefficient of $x_t$** from both terms of $\tilde{\mu}_t$:

$$\frac{\beta_t}{\sqrt{\alpha_t}\left(1-\bar{\alpha}_t\right)} + \frac{\sqrt{\alpha_t}\left(1-\bar{\alpha}_{t-1}\right)}{1-\bar{\alpha}_t} = \frac{\beta_t + \alpha_t\left(1-\bar{\alpha}_{t-1}\right)}{\sqrt{\alpha_t}\left(1-\bar{\alpha}_t\right)}$$

(putting both over the common denominator $\sqrt{\alpha_t}\left(1-\bar{\alpha}_t\right)$, which multiplies the second numerator by $\sqrt{\alpha_t} \cdot \sqrt{\alpha_t} = \alpha_t$).

Simplify the numerator with the two identities $\alpha_t + \beta_t = 1$ and $\alpha_t\, \bar{\alpha}_{t-1} = \bar{\alpha}_t$:

$$\beta_t + \alpha_t - \alpha_t\, \bar{\alpha}_{t-1} = 1 - \bar{\alpha}_t$$

So the $x_t$ coefficient is

$$\frac{1-\bar{\alpha}_t}{\sqrt{\alpha_t}\left(1-\bar{\alpha}_t\right)} = \frac{1}{\sqrt{\alpha_t}}$$

Putting the $x_t$ piece and the $\epsilon$ piece together:

$$\tilde{\mu}_t\!\left(x_t,\ \tfrac{1}{\sqrt{\bar{\alpha}_t}}\left(x_t - \sqrt{1-\bar{\alpha}_t}\,\epsilon\right)\right) = \frac{1}{\sqrt{\alpha_t}}\, x_t - \frac{\beta_t}{\sqrt{\alpha_t}\sqrt{1-\bar{\alpha}_t}}\, \epsilon = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\, \epsilon\right)$$

Plug this back into Eq. (9) and we have exactly the paper's Eq. (10):

$$L_{t-1} - C = \mathbb{E}_{x_0, \epsilon}\!\left[\frac{1}{2\sigma_t^2}\left\|\frac{1}{\sqrt{\alpha_t}}\left(x_t(x_0,\epsilon) - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\, \epsilon\right) - \mu_\theta\!\left(x_t(x_0,\epsilon),\ t\right)\right\|^2\right] \qquad (10)$$

### Numerical sanity check of the collapse

Take $t = 2$ with the linear schedule numbers from Section 21, and pick $x_t = 0.8$, $\epsilon = 0.3$ (one coordinate). Then $x_0 = \frac{1}{\sqrt{\bar{\alpha}_2}}\left(0.8 - \sqrt{1-\bar{\alpha}_2} \times 0.3\right) = 0.795639$.

- Eq. (7) directly: $\tilde{\mu}_2 = \frac{\sqrt{\bar{\alpha}_1}\, \beta_2}{1-\bar{\alpha}_2}(0.795639) + \frac{\sqrt{\alpha_2}\left(1-\bar{\alpha}_1\right)}{1-\bar{\alpha}_2}(0.8) = 0.7976218$
- Eq. (10)'s inner formula: $\frac{1}{\sqrt{\alpha_2}}\left(0.8 - \frac{\beta_2}{\sqrt{1-\bar{\alpha}_2}} \times 0.3\right) = 0.7976218$

Same number to 7 decimal places — the two forms are the same quantity, just written with different inputs.

### What Eq. (10) is telling us

The target that $\mu_\theta(x_t, t)$ must hit is

$$\frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\, \epsilon\right)$$

and this expression only involves things that are either **already given to the network** ($x_t$ is its input, $\alpha_t$, $\beta_t$, $\bar{\alpha}_t$ are known schedule numbers) or **the noise $\epsilon$**. So the *only* unknown part of the target is $\epsilon$ itself. That is the punchline this whole section has been building to: instead of asking the network to predict the mean $\tilde{\mu}_t$, we can ask it to predict just the noise $\epsilon$, and compute the mean from it by the fixed formula above. That is the $\epsilon$-prediction parameterization, Eq. (11) — the subject of the next part of these notes.

---

*End of this part — coverage now runs through Eq. (10). Next: Eq. (11), the $\epsilon$-parameterization $\mu_\theta = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\, \epsilon_\theta(x_t, t)\right)$, the sampling step, Eq. (12), and the connection to denoising score matching.*